# ANEEL RAG — Etapa 1: Ingestão e Download de PDFs
### Grupo de Estudos NLP · Desafio de Agentes 2025

**O que este notebook faz:**
1. Monta o Google Drive e cria a estrutura de pastas do projeto
2. Carrega os 3 JSONs de metadados e constrói um manifesto completo (27.039 entradas)
3. Faz análise adicional pré-download (URLs problemáticas, arquivos não-PDF, prioridades)
4. Baixa todos os arquivos de forma assíncrona com checkpoint automático
5. Gera relatório final de integridade

**Se a sessão cair:** execute novamente a partir da Célula 5 — o checkpoint retoma do ponto onde parou.

---
**Estrutura criada no Drive:**
```
aneel_rag/
├── raw_pdfs/    2016/ 2021/ 2022/   ← PDFs baixados
├── raw_other/   2016/ 2021/ 2022/   ← HTML, ZIP, XLSX
├── metadata/                         ← JSONs + manifesto.csv
└── logs/                             ← download_log.csv + erros
```

## Célula 1 — Instalação de dependências

In [ ]:
!pip install aiohttp aiofiles tqdm pandas --quiet
print("✓ Dependências instaladas")

!pip install nest_asyncio --quiet

import nest_asyncio
nest_asyncio.apply()  # permite asyncio.run() dentro de loop já existente

✓ Dependências instaladas


## Célula 2 — Montar Google Drive e criar estrutura de pastas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Raiz do projeto (ajuste se necessário) ──────────────────────────────────
BASE = Path('/content/drive/MyDrive/aneel_rag')

DIRS = {
    'raw_pdfs':  BASE / 'raw_pdfs',
    'raw_other': BASE / 'raw_other',
    'metadata':  BASE / 'metadata',
    'logs':      BASE / 'logs',
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

for year in ['2016', '2021', '2022']:
    (DIRS['raw_pdfs']  / year).mkdir(exist_ok=True)
    (DIRS['raw_other'] / year).mkdir(exist_ok=True)

print("✓ Estrutura criada em:", BASE)
for name, path in DIRS.items():
    print(f"  {name:<12}: {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Estrutura criada em: /content/drive/MyDrive/aneel_rag
  raw_pdfs    : /content/drive/MyDrive/aneel_rag/raw_pdfs
  raw_other   : /content/drive/MyDrive/aneel_rag/raw_other
  metadata    : /content/drive/MyDrive/aneel_rag/metadata
  logs        : /content/drive/MyDrive/aneel_rag/logs


## Célula 3 — Fazer upload dos JSONs

Faça upload dos 3 arquivos JSON usando o painel de arquivos do Colab (ícone de pasta à esquerda),
ou suba direto para a pasta `aneel_rag/metadata/` no Drive antes de executar.

Os nomes esperados:
- `biblioteca_aneel_gov_br_legislacao_2016_metadados.json`
- `biblioteca_aneel_gov_br_legislacao_2021_metadados.json`
- `biblioteca_aneel_gov_br_legislacao_2022_metadados.json`

In [ ]:
import shutil, os

# Se fizer upload direto no Colab (não no Drive), mova para o Drive:
for year in ['2016', '2021', '2022']:
    fname = f'biblioteca_aneel_gov_br_legislacao_{year}_metadados.json'
    src   = Path(f'/content/{fname}')
    dst   = DIRS['metadata'] / fname
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f"✓ Copiado: {fname}")
    elif dst.exists():
        print(f"  Já existe no Drive: {fname}")
    else:
        print(f"  ⚠ Não encontrado: {fname}")

JSON_PATHS = {
    yr: DIRS['metadata'] / f'biblioteca_aneel_gov_br_legislacao_{yr}_metadados.json'
    for yr in ['2016', '2021', '2022']
}

# Verificar que existem
for yr, path in JSON_PATHS.items():
    status = "✓" if path.exists() else "✗ FALTANDO"
    print(f"  {status}  {yr}: {path.name}")

  Já existe no Drive: biblioteca_aneel_gov_br_legislacao_2016_metadados.json
  Já existe no Drive: biblioteca_aneel_gov_br_legislacao_2021_metadados.json
  Já existe no Drive: biblioteca_aneel_gov_br_legislacao_2022_metadados.json
  ✓  2016: biblioteca_aneel_gov_br_legislacao_2016_metadados.json
  ✓  2021: biblioteca_aneel_gov_br_legislacao_2021_metadados.json
  ✓  2022: biblioteca_aneel_gov_br_legislacao_2022_metadados.json


## Célula 4 — Funções auxiliares (normalização e fixup de URLs)

In [ ]:
import json
import re
import pandas as pd
from collections import Counter

def normalize_tipo_pdf(raw: str) -> str:
    """Normaliza o campo 'tipo' de 757 variações para 21 categorias limpas."""
    t  = (raw or '').strip().rstrip(':').strip()
    tl = t.lower().replace(' ', '').replace('_', '').replace('-', '')
    if re.search(r'tex+to\s*i[nt][te]?[eg][er]?[ra]*[la]?[l]?|^integral$|^tex+to\s*:?$', t, re.I):
        return 'texto_integral'
    if re.search(r'voto.{0,15}(vista|view)', t, re.I):    return 'voto_vista'
    if re.search(r'voto.{0,15}divergen', t, re.I):         return 'voto_divergente'
    if re.search(r'voto.{0,15}(resumo|condutor|complemento|separado)', t, re.I): return 'voto_complementar'
    if re.search(r'voto.{0,15}aviso', t, re.I):            return 'voto_aviso'
    if re.search(r'^voto', t, re.I):                       return 'voto'
    if re.search(r'nota\s*t[eé]cnica|^NT[\s\.\-]', t, re.I): return 'nota_tecnica'
    if re.search(r'decis[aã]o\s*judicial', t, re.I):      return 'decisao_judicial'
    if re.search(r'decis[aã]o', t, re.I):                  return 'decisao'
    if re.search(r'sub.?m[oó]dulo', t, re.I):              return 'anexo_submodulo'
    if re.search(r'anexo', t, re.I):                       return 'anexo'
    if re.search(r'exposi.{0,5}motivos', t, re.I):         return 'exposicao_motivos'
    if re.search(r'mem[oó]ria\s*de\s*c[aá]lculo', t, re.I): return 'memoria_calculo'
    if re.search(r'base\s*de\s*da', t, re.I):            return 'base_dados'
    if re.search(r'planilha|simulador|programa\s*nodal|gloss[aá]rio|pleito', t, re.I): return 'planilha_calculo'
    if re.search(r'regi[aã]o\s*(norte|sul|nordeste|sudeste|centro)', t, re.I): return 'anexo_regional'
    if not t or tl in ('pdf', 'site1', ':'):               return 'sem_tipo'
    return 'outro'

def fix_url(raw: str) -> str:
    """Corrige URLs malformadas (duplo http://, espaços internos)."""
    url = (raw or '').strip()
    url = url.replace(' ', '')
    url = re.sub(r'https?://\s*https?://', 'http://', url)
    return url

def get_tipo_sigla(titulo: str) -> str:
    if ' - ' in titulo: return titulo.split(' - ')[0].strip()
    return titulo.split()[0] if titulo else 'DESCONHECIDO'

def get_file_type(ext: str) -> str:
    ext = ext.lower().strip()
    if ext in ('pdf', ''):         return 'pdf'
    if ext in ('html', 'htm'):     return 'html'
    if ext in ('zip', 'rar'):      return 'archive'
    if ext in ('xlsx', 'xlsm', 'xls'): return 'spreadsheet'
    return 'other'

print("✓ Funções auxiliares carregadas")

✓ Funções auxiliares carregadas


## Célula 5 — Construir manifesto completo de todos os arquivos

In [ ]:
records   = []
seen_urls = set()

for year, path in JSON_PATHS.items():
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    for date_str, day_data in data.items():
        for doc in day_data.get('registros', []):
            titulo   = doc.get('titulo', '') or ''
            sigla    = get_tipo_sigla(titulo)
            situacao = (doc.get('situacao') or '').replace('Situação:', '').strip()
            is_active = situacao in ('NÃO CONSTA REVOGAÇÃO EXPRESSA', '')
            ementa   = (doc.get('ementa') or '').strip()
            autor    = (doc.get('autor') or '').strip()
            assunto  = (doc.get('assunto') or '').replace('Assunto:', '').strip()
            pub      = (doc.get('publicacao') or '').replace('Publicação:', '').strip()
            assin    = (doc.get('assinatura') or '').replace('Assinatura:', '').strip()

            for pdf in (doc.get('pdfs') or []):
                raw_url  = pdf.get('url', '') or ''
                arquivo  = (pdf.get('arquivo') or '').strip()
                tipo_raw = (pdf.get('tipo') or '').strip()
                baixado  = pdf.get('baixado', False)

                url      = fix_url(raw_url)
                ext      = arquivo.rsplit('.', 1)[-1].lower().strip() if '.' in arquivo else ''
                tipo_norm = normalize_tipo_pdf(tipo_raw)
                file_type = get_file_type(ext)
                is_dup    = url in seen_urls
                seen_urls.add(url)
                doc_id    = arquivo.rsplit('.', 1)[0] if '.' in arquivo else arquivo

                records.append({
                    'doc_id':           doc_id,
                    'arquivo':          arquivo,
                    'ano':              year,
                    'data_publicacao':  pub,
                    'data_assinatura':  assin,
                    'titulo':           titulo,
                    'sigla':            sigla,
                    'autor':            autor,
                    'assunto':          assunto,
                    'situacao':         situacao,
                    'is_active':        is_active,
                    'ementa':           ementa,
                    'tipo_pdf_raw':     tipo_raw,
                    'tipo_pdf_norm':    tipo_norm,
                    'is_principal':     tipo_norm == 'texto_integral',
                    'file_type':        file_type,
                    'extensao':         ext,
                    'url_original':     raw_url,
                    'url':              url,
                    'baixado_json':     baixado,
                    'is_duplicata':     is_dup,
                    'download_status':  'pendente',
                    'local_path':       '',
                    'tamanho_bytes':    0,
                    'erro':             '',
                })

manifest = pd.DataFrame(records)
manifest_path = DIRS['metadata'] / 'manifesto_completo.csv'
manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')

print(f"✓ Manifesto construído: {len(manifest):,} entradas")
print(f"  Salvo em: {manifest_path}")
print()
print("Distribuição por tipo de arquivo:")
print(manifest.file_type.value_counts().to_string())
print()
print("Distribuição por ano:")
print(manifest.groupby('ano').size().to_string())
print()
print(f"Duplicatas URL     : {manifest.is_duplicata.sum():,}")
print(f"Docs inativos      : {(~manifest.is_active).sum():,}")
print(f"Com ementa         : {manifest.ementa.str.len().gt(0).sum():,} ({manifest.ementa.str.len().gt(0).mean()*100:.1f}%)")

✓ Manifesto construído: 27,039 entradas
  Salvo em: /content/drive/MyDrive/aneel_rag/metadata/manifesto_completo.csv

Distribuição por tipo de arquivo:
file_type
pdf            26785
html             188
archive           49
spreadsheet       17

Distribuição por ano:
ano
2016     6279
2021     9624
2022    11136

Duplicatas URL     : 14
Docs inativos      : 1,904
Com ementa         : 24,991 (92.4%)


## Célula 6 — Análise adicional pré-download

In [ ]:
print("=" * 58)
print("ANÁLISE ADICIONAL — Pontos de atenção pré-download")
print("=" * 58)

# 1. URLs ainda problemáticas após normalização
mal = manifest[manifest.url.str.contains(r'[^\x00-\x7F]|  |\s', regex=True, na=False)]
print(f"\n1. URLs problemáticas após normalização   : {len(mal)}")
for _, row in mal.iterrows():
    print(f"   repr: {repr(row['url'][:80])}")

# 2. Documentos sem Texto Integral
docs_com_ti = set(manifest[manifest.tipo_pdf_norm == 'texto_integral']['titulo'])
docs_sem_ti = manifest[~manifest['titulo'].isin(docs_com_ti)]['titulo'].unique()
print(f"\n2. Docs sem Texto Integral mapeado        : {len(docs_sem_ti)}")
print(f"   (só têm Voto, NT ou Anexo)")
if len(docs_sem_ti) > 0:
    for d in docs_sem_ti[:3]: print(f"   ex: {d}")

# 3. Arquivos não-PDF por tipo de documento
nao_pdf = manifest[manifest.file_type != 'pdf']
print(f"\n3. Arquivos não-PDF por tipo de documento:")
cross = pd.crosstab(nao_pdf.sigla, nao_pdf.file_type)
print(cross.to_string())

# 4. Planalto.gov.br
planalto = manifest[manifest.url.str.contains('planalto.gov.br', na=False)]
print(f"\n4. Arquivos no planalto.gov.br            : {len(planalto)}")
print(f"   (leis e decretos referenciados pela ANEEL)")

# 5. Plano de prioridade
pdf_ativos   = manifest[(manifest.file_type == 'pdf') & manifest.is_active & ~manifest.is_duplicata]
pdf_inativos = manifest[(manifest.file_type == 'pdf') & ~manifest.is_active & ~manifest.is_duplicata]
print(f"\n5. Plano de download por prioridade:")
print(f"   P1 — PDFs ativos únicos      : {len(pdf_ativos):>6,}  ← MAIS IMPORTANTE")
print(f"   P2 — PDFs inativos únicos    : {len(pdf_inativos):>6,}  ← manter para histórico")
print(f"   P3 — HTML / HTM              : {len(manifest[manifest.file_type=='html']):>6,}")
print(f"   P4 — ZIP / RAR               : {len(manifest[manifest.file_type=='archive']):>6,}")
print(f"   P5 — XLSX / XLSM             : {len(manifest[manifest.file_type=='spreadsheet']):>6,}")
print(f"   Skip duplicatas              : {manifest.is_duplicata.sum():>6,}")

# 6. Top siglas com mais arquivos para download
print(f"\n6. Top 8 tipos de documento por volume:")
sigla_stats = manifest[~manifest.is_duplicata].groupby('sigla').agg(
    total=('doc_id', 'count'),
    ativos=('is_active', 'sum'),
    pdfs=('file_type', lambda x: (x == 'pdf').sum())
).sort_values('total', ascending=False).head(8)
print(sigla_stats.to_string())

ANÁLISE ADICIONAL — Pontos de atenção pré-download

1. URLs problemáticas após normalização   : 0

2. Docs sem Texto Integral mapeado        : 7
   (só têm Voto, NT ou Anexo)
   ex: DSP - DESPACHO 3005/2016
   ex: DSP - DESPACHO 442/2016
   ex: DSP - DESPACHO 1729/2021

3. Arquivos não-PDF por tipo de documento:
file_type  archive  html  spreadsheet
sigla                                
DEC              0    31            0
DSP             10     0            5
LEI              0     3            0
PRT              1     0            7
REH             24     0            3
REN             14   154            2

4. Arquivos no planalto.gov.br            : 34
   (leis e decretos referenciados pela ANEEL)

5. Plano de download por prioridade:
   P1 — PDFs ativos únicos      : 24,964  ← MAIS IMPORTANTE
   P2 — PDFs inativos únicos    :  1,807  ← manter para histórico
   P3 — HTML / HTM              :    188
   P4 — ZIP / RAR               :     49
   P5 — XLSX / XLSM             :     17
 

## Célula 7 — Download assíncrono com checkpoint automático

In [ ]:
import asyncio
import aiohttp
import aiofiles
import time
from tqdm.auto import tqdm

CONCURRENCY    = 8
TIMEOUT_SEC    = 45
MAX_RETRIES    = 3
CHECKPOINT_EVERY = 100
LOG_PATH = DIRS['logs'] / 'download_log.csv'

def get_local_path(row) -> Path:
    year  = str(row['ano'])
    fname = row['arquivo'] if row['arquivo'] else f"{row['doc_id']}.{row['extensao'] or 'bin'}"
    if row['file_type'] == 'pdf':
        return DIRS['raw_pdfs'] / year / fname
    return DIRS['raw_other'] / year / fname

async def download_file(session, sem, row, pbar) -> dict:
    url    = row['url']
    path   = get_local_path(row)
    result = {'doc_id': row['doc_id'], 'url': url, 'arquivo': row['arquivo'],
              'status': 'erro', 'tamanho_bytes': 0, 'erro': '', 'timestamp': ''}

    # Pular se já existe localmente
    if path.exists() and path.stat().st_size > 512:
        result['status']        = 'ok'
        result['tamanho_bytes'] = path.stat().st_size
        result['timestamp']     = 'existia'
        pbar.update(1)
        return result

    async with sem:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=TIMEOUT_SEC)) as resp:
                    if resp.status != 200:
                        result['erro'] = f"HTTP {resp.status}"
                        await asyncio.sleep(2)
                        continue
                    content = await resp.read()
                    if len(content) < 512:
                        result['erro'] = f"muito pequeno ({len(content)}b)"
                        break
                    if row['file_type'] == 'pdf' and not content.startswith(b'%PDF'):
                        result['erro'] = "não começa com %PDF"
                        break
                    async with aiofiles.open(path, 'wb') as f:
                        await f.write(content)
                    result['status']        = 'ok'
                    result['tamanho_bytes'] = len(content)
                    result['timestamp']     = time.strftime('%Y-%m-%dT%H:%M:%S')
                    break
            except asyncio.TimeoutError:
                result['erro'] = f"timeout (t{attempt})"
                await asyncio.sleep(2 ** attempt)
            except Exception as e:
                result['erro'] = str(e)[:100]
                await asyncio.sleep(2 ** attempt)

    pbar.update(1)
    return result

async def download_all_async(rows):
    sem       = asyncio.Semaphore(CONCURRENCY)
    results   = []
    log_buffer = []
    connector = aiohttp.TCPConnector(limit=CONCURRENCY + 4, ssl=False)
    headers   = {'User-Agent': 'Mozilla/5.0 (ANEEL-RAG-Research/1.0)'}

    async with aiohttp.ClientSession(connector=connector, headers=headers) as session:
        with tqdm(total=len(rows), desc='Download', unit='arq', dynamic_ncols=True) as pbar:
            tasks = [download_file(session, sem, row, pbar) for row in rows]
            for coro in asyncio.as_completed(tasks):
                r = await coro
                results.append(r)
                log_buffer.append(r)
                if len(log_buffer) >= CHECKPOINT_EVERY:
                    df = pd.DataFrame(log_buffer)
                    df.to_csv(LOG_PATH, mode='a', header=not LOG_PATH.exists(),
                              index=False, encoding='utf-8-sig')
                    log_buffer.clear()

    if log_buffer:
        df = pd.DataFrame(log_buffer)
        df.to_csv(LOG_PATH, mode='a', header=not LOG_PATH.exists(),
                  index=False, encoding='utf-8-sig')
    return results

print("✓ Funções de download carregadas")
print(f"  Concorrência: {CONCURRENCY} downloads simultâneos")
print(f"  Timeout por arquivo: {TIMEOUT_SEC}s")
print(f"  Tentativas por arquivo: {MAX_RETRIES}")
print(f"  Checkpoint a cada: {CHECKPOINT_EVERY} arquivos")

✓ Funções de download carregadas
  Concorrência: 8 downloads simultâneos
  Timeout por arquivo: 45s
  Tentativas por arquivo: 3
  Checkpoint a cada: 100 arquivos


## Célula 8 — Executar download

> **⚠ Esta célula pode levar várias horas dependendo da conexão.**
> Se a sessão cair, execute novamente — o checkpoint garante que os arquivos já baixados sejam ignorados.
>
> Você pode monitorar o progresso em: `aneel_rag/logs/download_log.csv`

In [ ]:
# Recarregar manifesto e verificar o que já foi baixado
mf = pd.read_csv(DIRS['metadata'] / 'manifesto_completo.csv', encoding='utf-8-sig')

# Já OK no log
ja_ok = set()
if LOG_PATH.exists():
    log_exist = pd.read_csv(LOG_PATH, encoding='utf-8-sig')
    ja_ok = set(log_exist[log_exist['status'] == 'ok']['url'].tolist())
    print(f"✓ Log existente: {len(ja_ok):,} arquivos já OK")

# Filtrar pendentes (não duplicatas, não já baixados)
pendentes = mf[(~mf['url'].isin(ja_ok)) & (~mf['is_duplicata'])].copy()

# Ordenar por prioridade: texto_integral ativo primeiro
def prioridade(row):
    if row['is_principal'] and row['is_active']:    return 0
    if row['is_principal'] and not row['is_active']: return 1
    if row['file_type'] == 'pdf':                   return 2
    if row['file_type'] == 'html':                  return 3
    return 4

pendentes['_pri'] = pendentes.apply(prioridade, axis=1)
pendentes = pendentes.sort_values('_pri').drop(columns=['_pri'])

print(f"\nResumo do download:")
print(f"  Total no manifesto     : {len(mf):,}")
print(f"  Já baixados (ok)       : {len(ja_ok):,}")
print(f"  Duplicatas (skip)      : {mf.is_duplicata.sum():,}")
print(f"  A baixar agora         : {len(pendentes):,}")

if len(pendentes) == 0:
    print("\n✓ Todos os arquivos já foram baixados!")
else:
    print(f"\nIniciando download...")
    rows = pendentes.to_dict('records')
    results = asyncio.run(download_all_async(rows))

    ok    = sum(1 for r in results if r['status'] == 'ok')
    erros = sum(1 for r in results if r['status'] != 'ok')
    total_mb = sum(r['tamanho_bytes'] for r in results if r['status'] == 'ok') / (1024*1024)

    print(f"\n{'='*45}")
    print(f"Download concluído")
    print(f"  OK     : {ok:,}")
    print(f"  Erros  : {erros:,}")
    print(f"  Total  : {total_mb:.1f} MB baixados agora")

    # Atualizar manifesto
    log_all = pd.read_csv(LOG_PATH, encoding='utf-8-sig')
    ok_urls = set(log_all[log_all['status'] == 'ok']['url'])
    def calc_status(row):
        if row['url'] in ok_urls:      return 'ok'
        if row['is_duplicata']:        return 'skip_dup'
        return 'pendente'
    mf['download_status'] = mf.apply(calc_status, axis=1)
    mf.to_csv(DIRS['metadata'] / 'manifesto_completo.csv', index=False, encoding='utf-8-sig')
    print(f"  Manifesto atualizado")

✓ Log existente: 1 arquivos já OK

Resumo do download:
  Total no manifesto     : 27,039
  Já baixados (ok)       : 1
  Duplicatas (skip)      : 14
  A baixar agora         : 27,024

Iniciando download...


Download:   0%|          | 0/27024 [00:00<?, ?arq/s]


Download concluído
  OK     : 33
  Erros  : 26,991
  Total  : 1.8 MB baixados agora
  Manifesto atualizado


## Célula 9 — Relatório final e verificação de integridade

In [ ]:
mf = pd.read_csv(DIRS['metadata'] / 'manifesto_completo.csv', encoding='utf-8-sig')

ok     = mf[mf.download_status == 'ok']
erro   = mf[mf.download_status.isin(['erro', 'pendente'])]
skip   = mf[mf.is_duplicata]

print("=" * 55)
print("RELATÓRIO FINAL — Etapa 1: Ingestão")
print("=" * 55)
print(f"\nTotal no manifesto         : {len(mf):>6,}")
print(f"  Baixados com sucesso     : {len(ok):>6,}")
print(f"  Com erro / pendentes     : {len(erro):>6,}")
print(f"  Duplicatas (skip)        : {len(skip):>6,}")

if LOG_PATH.exists():
    log = pd.read_csv(LOG_PATH, encoding='utf-8-sig')
    total_gb = log[log.status == 'ok']['tamanho_bytes'].sum() / (1024**3)
    print(f"\nTamanho total baixado      : {total_gb:.2f} GB")

print(f"\nPor tipo de arquivo (OK):")
for ft in ok.file_type.unique():
    n = len(ok[ok.file_type == ft])
    print(f"  {ft:<14}: {n:>5,}")

print(f"\nPor ano — PDFs OK:")
pdfs_ok = ok[ok.file_type == 'pdf']
for yr in ['2016', '2021', '2022']:
    n = len(pdfs_ok[pdfs_ok.ano == yr])
    print(f"  {yr}: {n:,}")

if len(erro) > 0:
    print(f"\nArquivos com erro: {len(erro)}")
    print(f"  Lista salva em: {DIRS['logs'] / 'erros_download.csv'}")
    erro.to_csv(DIRS['logs'] / 'erros_download.csv', index=False, encoding='utf-8-sig')

print(f"\n{'='*55}")
print(f"✓ Etapa 1 concluída!")
print(f"\nPróxima etapa:")
print(f"  Etapa 2 — Extração de texto com Docling + pymupdf4llm")
print(f"  Entrada : {DIRS['raw_pdfs']}")
print(f"  Manifesto: {DIRS['metadata'] / 'manifesto_completo.csv'}")
print(f"{'='*55}")

RELATÓRIO FINAL — Etapa 1: Ingestão

Total no manifesto         : 27,039
  Baixados com sucesso     :     34
  Com erro / pendentes     : 26,991
  Duplicatas (skip)        :     14

Tamanho total baixado      : 0.00 GB

Por tipo de arquivo (OK):
  html          :    34

Por ano — PDFs OK:
  2016: 0
  2021: 0
  2022: 0

Arquivos com erro: 26991
  Lista salva em: /content/drive/MyDrive/aneel_rag/logs/erros_download.csv

✓ Etapa 1 concluída!

Próxima etapa:
  Etapa 2 — Extração de texto com Docling + pymupdf4llm
  Entrada : /content/drive/MyDrive/aneel_rag/raw_pdfs
  Manifesto: /content/drive/MyDrive/aneel_rag/metadata/manifesto_completo.csv
